# Solar EoL Transportation Cost Estimator

This project focuses on estimating the cost of transporting EoL solar PV modules from all power stations in Australia to nearest supplier/buyer of each recoverable material. 

This project also show the difference in estimated transportation cost of Conventional and Mobile recycling method.

Furthermore, this project also include the estimated monetary value of recoverable materials in order to show how many modules should be recycled in one go so we reach a break-even point or obtain profit.

#### Assumptions

To simplify the calculation, we decided to use the following assumptions.
1. "Photovoltaic solar panels consist of 95% recyclable materials" (Clean Energy Council, 2025). This means almost 100% of PV components can be recycled and sold. Hence, our assumption is 100% of PV materials can be recycled and would be available to be sold to supplier.
2. The composition of each material in each PV module is as follows.
3. For estimating monetary value and converting different units, we use the following estimation. \
    30kg/panel \
    0.6kW/panel \
    50kg/kW \
`10$/kw = worth of panel power` \
`10$/50kg = 0.2$/kg = worth of panel waste`
4. The capacity of solar EoL panel is 300 W per module.
5. Truck capacity for transporting EoL modules: 7,000 kg. Truck dimension: 2.2 m x 6.2 m x 2.4 m.

#### Install necessary packages

In [1]:
!pip install beautifulsoup4 pandas folium

In [8]:
!pip install googlemaps

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for googlemaps: filename=googlemaps-4.10.0-py3-none-any.whl size=40749 sha256=9b567f26d5118af86f7c1bed035e8e9a893fedd8f113a5c6f1fc34afe203c7e1
  Stored in directory: c:\users\lenovo\appdata\local\pip\cache\wheels\76\2a\24\5993a7b77c9a37b86f415096436a448c1babdd132066bdcb31
Successfully built googlemaps


  DEPRECATION: Building 'googlemaps' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'googlemaps'. Discussion can be found at https://github.com/pypa/pip/issues/6334


#### Extract position of all solar panel power stations

In [3]:
import pandas as pd
from bs4 import BeautifulSoup
import re

# 1. Load the HTML content from your saved .txt file
with open('power_stations.txt', 'r', encoding='utf-8') as file:
    soup = BeautifulSoup(file.read(), 'html.parser')

# 2. Find the table containing the data
table = soup.find('table', class_='power-stations-data-table')
data = []

# 3. Loop through every row in the table body
for row in table.find('tbody').find_all('tr'):
    cols = row.find_all('td')
    
    # Ensure the row has the correct number of columns
    if len(cols) >= 8:
        name = cols[0].text.strip()
        size = cols[3].text.strip()
        
        # Extract the raw link exactly as it is written in your HTML
        link_tag = cols[7].find('a')
        link = link_tag['href'] if link_tag and 'href' in link_tag.attrs else None
        
        lat, lon = None, None
        
        # 4. Use Regex to extract the coordinates directly from the link string
        if link:
            # This looks for the '@' symbol, grabs the negative/positive decimal, 
            # skips the comma, and grabs the second decimal.
            coords_match = re.search(r'@([-]?\d+\.\d+),([-]?\d+\.\d+)', link)
            
            if coords_match:
                lat = float(coords_match.group(1))
                lon = float(coords_match.group(2))
        
        # Append the extracted info to our list
        data.append({
            'Name': name,
            'Size (kW)': size,
            'Link': link,
            'Latitude': lat,
            'Longitude': lon
        })

In [4]:
# Convert to a Pandas DataFrame
df = pd.DataFrame(data)
df

,Name,Size (kW),Link,Latitude,Longitude
0,100 Harris Street Pyrmont,199.9,"https://www.google.com/maps/@-33.86835036,151....",-33.868350,151.193742
1,110 Somersby Falls Rd,199.1,"https://www.google.com/maps/@-33.410577,151.27...",-33.410577,151.279734
2,115 Frederick Street,292.0,"https://www.google.com/maps/@-27.387098,153.07...",-27.387098,153.075908
3,1182 Old Port Rd Power Station,199.7,"https://www.google.com/maps/@-34.86368301,138....",-34.863683,138.508196
4,123 Sippy Downs,199.6,"https://www.google.com/maps/@-26.713876,153.06...",-26.713876,153.061123
...,...,...,...,...,...
2873,Zammit Ham 2,281.1,"https://www.google.com/maps/@-33.798161,150.95...",-33.798161,150.956296
2874,ZENEXUS,250.3,"https://www.google.com/maps/@-27.58794,152.878...",-27.587940,152.878980
2875,Zerella Virginia,997.0,"https://www.google.com/maps/@-34.6397,138.5889...",-34.639700,138.588900
2876,ZEUSAPPOLLO SOLAR TC Do,141.6,"https://www.google.com/maps/@-31.3514,115.5661...",-31.351400,115.566100


#### List of Recycling Hub

#### List of Supplier/Buyer

## Conventional Recycling

### Number of Panels in Each Truck

In [2]:
# Based on the following assumptions:
truck_volume_limit = 28 # cubic meters
truck_weight_limit = 7000 # kg
truck_w, truck_l, truck_h = 2.2, 6.2, 2.4 # meters
module_capacity_w = 300 # Watts
module_w, module_l, module_h = 1.0, 1.7, 0.035 # meters

# Additional assumption based on previous parameter: 50kg/kW
module_weight = (module_capacity_w / 1000) * 50 # 15 kg
module_volume = module_w * module_l * module_h # 0.0595 cubic meters

# Capacity of Truck limited by volume, weight, and physical space
print(f"How many panels can one standard 5-ton truck carry?")

# Number of panels limit by volume
panels_by_volume = int(truck_volume_limit // module_volume)
print(f"Limit by pure volume: {panels_by_volume} panels")

# Number of panels limit by weight
panels_by_weight = int(truck_weight_limit // module_weight)
print(f"Limit by pure weight: {panels_by_weight} panels")

# Number of panels limit by physical space
stacks_wide = int(truck_w // module_w)
stacks_long = int(truck_l // module_l)
panels_high = int(truck_h // module_h)

panels_by_space = stacks_wide * stacks_long * panels_high
print(f"Limit by physical space: {panels_by_space} panels")

# Conclusion
print(f"\nConclusion: Limited by physical space, the truck can carry {panels_by_space} panels.")

How many panels can one standard 5-ton truck carry?
Limit by pure volume: 470 panels
Limit by pure weight: 466 panels
Limit by physical space: 408 panels

Conclusion: Limited by physical space, the truck can carry 408 panels.


### Number of Truck Needed

In [4]:
import math

def calculate_eol_logistics():
    print("--- Solar Module Number of Trucks Calculator ---")
    print("Select the unit to input your data:")
    print("1: Kilowatts (kW)")
    print("2: Kilograms (kg)")
    print("3: Number of Panels")
    
    choice = input("Enter 1, 2, or 3: ")
    
    if choice not in ['1', '2', '3']:
        print("Invalid selection. Please restart.")
        return

    try:
        value = float(input("Enter the value: "))
    except ValueError:
        print("Invalid number. Please enter a numerical value.")
        return

    # Base Assumptions (300W Module)
    kw_per_panel = 0.300
    kg_per_panel = 15.0
    panels_per_truck = 408 # Maximum physical limit for 5-ton standard truck

    # Conversion Logic & Dynamic Input String
    if choice == '1': # User inputted kW
        total_kw = value
        total_panels = total_kw / kw_per_panel
        total_kg = total_panels * kg_per_panel
        input_display = f"{value:,.2f} kW of solar capacity"
        
    elif choice == '2': # User inputted kg
        total_kg = value
        total_panels = total_kg / kg_per_panel
        total_kw = total_panels * kw_per_panel
        input_display = f"{value:,.2f} kg of solar modules"
        
    elif choice == '3': # User inputted Number of Panels
        total_panels = value
        total_kw = total_panels * kw_per_panel
        total_kg = total_panels * kg_per_panel
        input_display = f"{value:,.0f} panels"

    # Truck Calculation
    trucks_needed = math.ceil(total_panels / panels_per_truck)

    # Output Results
    print("\n--- Converted Values ---")
    print(f"Capacity:      {total_kw:,.2f} kW")
    print(f"Total Mass:    {total_kg:,.2f} kg")
    print(f"Total Panels:  {total_panels:,.0f} panels")
    
    print("\n--- Transport Requirements ---")
    print(f"To transport {input_display}, you will need {trucks_needed} truck(s).")

if __name__ == "__main__":
    calculate_eol_logistics()

--- Solar Module Number of Trucks Calculator ---
Select the unit to input your data:
1: Kilowatts (kW)
2: Kilograms (kg)
3: Number of Panels


Enter 1, 2, or 3:  1
Enter the value:  1066



--- Converted Values ---
Capacity:      1,066.00 kW
Total Mass:    53,300.00 kg
Total Panels:  3,553 panels

--- Transport Requirements ---
To transport 1,066.00 kW of solar capacity, you will need 9 truck(s).


### Estimate Transportation Cost

In [6]:
# 